# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
# using pypdf drawing from code provided on LangChain documentation

import pypdf
from langchain_core.documents import Document

reader = pypdf.PdfReader('documents/ai_report_2025.pdf')  

docs = [
    Document(
        page_content=page.extract_text() or "",
        metadata={"page": i}
    )
    for i, page in enumerate(reader.pages)    #looping through each page to read the text
]


In [3]:
# joining pages together

full_pdf_text =  ""
for page in docs:
    full_pdf_text += page.page_content + "\n"


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
import sys
sys.path.append('../05_src/')

from utils.clients import get_client
import os
from IPython.display import display, Markdown

os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client(use_gateway=True)


# OpenAI Gateway Setup Notes:
# AuthenticationError (401) was raised during generation task below.
# Troubleshooting with client.base_url showed that requests went to OpenAI directly instead of the gateway.
# Setting get_client(use_gateway=True) allows override of he default and forces it to use gateway instead.

In [5]:
from typing import Optional
from pydantic import BaseModel, Field
class PdfAnalysis(BaseModel):
    Author: str=Field(description="The author of the article")
    Title: str=Field(description="The title of the article")
    Relevance: str=Field(description="max one paragraph on why article is relevant for an AI professional in their professional development")
    Summary: str=Field(description="A concise summary no longer than 1000 tokens")
    Tone: str=Field(description="The tone used to produce the summary")
    InputTokens: Optional[int]=Field(description="The number of input tokens from response object")
    OutputTokens: Optional[int]=Field(description="The number of output tokens from response object")


# following code provided on structured_outputs jupyter notebook
# defining schema makes the model return structured output
# setting Optional for Input and Output tokens to allow return of null

In [6]:
# developer prompt
instructions = """
    You are a research assistant.
    Analyze the article and extract structured information.
    
    Return:
    - Author
    - Title
    - Relevance: a statement, maximum one paragraph, explaining why is the article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary
    - InputTokens: number of input tokens from response object
    - OutputTokens: number of input tokens from response object
    """

tone = "Formal Academic Writing"

#user prompt
prompt = f"""
    Analyze the following article.
    <article>
    {full_pdf_text}
    </article>
    
    Write the response in {tone}.
    
    """


In [7]:
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": instructions},
        {"role": "user", "content": prompt},
    ],
    text_format=PdfAnalysis,
)

result = response.output_parsed

result.InputTokens = response.usage.input_tokens
result.OutputTokens = response.usage.output_tokens

result 

# following code provided on structured_outputs jupyter notebook

PdfAnalysis(Author='MIT NANDA (Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari)', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This article is highly relevant for AI professionals as it delves into critical insights surrounding the adoption and implementation of Generative AI (GenAI) in organizations. Understanding the common pitfalls and success factors outlined in the report will inform practitioners and decision-makers on how to effectively leverage AI technologies, fostering both innovation and operational improvement in their respective enterprises.', Summary="The article 'The GenAI Divide: State of AI in Business 2025' presents findings from Project NANDA, illustrating a striking disparity known as the 'GenAI Divide'. Despite substantial investments in Generative AI, a staggering 95% of organizations report negligible returns, primarily attributed to ineffective implementations rather than technological limitations. The research, which encomp

In [8]:
#displaying result in easy-to-read Markdown format

display(Markdown(f"""
**Author:** {result.Author}

**Title:** {result.Title}

**Relevance:** {result.Relevance}

**Summary:** {result.Summary}

**Tone:** {result.Tone}

**Input Tokens:** {result.InputTokens}

**Output Tokens:** {result.OutputTokens}
"""))


**Author:** MIT NANDA (Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari)

**Title:** The GenAI Divide: State of AI in Business 2025

**Relevance:** This article is highly relevant for AI professionals as it delves into critical insights surrounding the adoption and implementation of Generative AI (GenAI) in organizations. Understanding the common pitfalls and success factors outlined in the report will inform practitioners and decision-makers on how to effectively leverage AI technologies, fostering both innovation and operational improvement in their respective enterprises.

**Summary:** The article 'The GenAI Divide: State of AI in Business 2025' presents findings from Project NANDA, illustrating a striking disparity known as the 'GenAI Divide'. Despite substantial investments in Generative AI, a staggering 95% of organizations report negligible returns, primarily attributed to ineffective implementations rather than technological limitations. The research, which encompasses a systematic review of over 300 AI initiatives, interviews with executives across 52 organizations, and survey data from 153 senior leaders, reveals that while tools like ChatGPT are widely adopted to enhance productivity, they seldom foster meaningful transformation in financial performance. Key findings include: (1) Misalignment between pilot implementations and operational workflows; (2) A prevalent preference for consumer-grade AI tools over customized enterprise solutions due to perceived reliability and user experience; (3) The emergence of a 'shadow AI economy' where employees utilize personal AI tools more effectively than official systems; (4) A significant investment bias toward sales and marketing functions over high-ROI back-office automation, which often yields better returns; and (5) An outlined path to success for organizations that demand deeply integrated and adaptive AI solutions. The article concludes with a clarion call for enterprises to reconsider their AI strategies, pivoting from static tool investments towards partnerships that emphasize customization and learning capabilities. As organizations approach an increasingly competitive landscape, the imperative to bridge the GenAI Divide could define their future operational capabilities and market positioning.

**Tone:** Formal and Analytical

**Input Tokens:** 11052

**Output Tokens:** 422


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [9]:
# importing necessary modules from DeepEval

from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams # using LLMTestCaseParams returned phase-out warning. SingleTurnParams was suggested as an alternative.
from deepeval.models import GPTModel

# setting up evaluation model
eval_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)


In [10]:
# creating a test case

test_case = LLMTestCase(input=full_pdf_text, actual_output=result.Summary)

#summarization evaluation
summarization_metric = SummarizationMetric(
    threshold=0.7,
    model=eval_model,
    async_mode=False,
    assessment_questions=[
        "Does the summary reflect the main topic of the report?",
        "Does the summary identify key trends mentioned in the report?",
        "Is the summary free of contradictions with the original text?",
        "Does the summary avoid adding facts not in the original report?",
        "Does the summary reflect the report's main conclusion?"

    ]
)



In [11]:
#clarity evaluation
clarity_metric = GEval(
    name="Clarity",
    model=eval_model,
    evaluation_steps=[
        "Is the response well organized and logically structured?"
        "Does the response use clear, concise, and direct language?",
        "Are technical terms explained or used appropriately?",
        "Are complex ideas presented ways that are easy to follow?",
        "Does the response maintain the same style and readability throughout?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

#tonality evaluation
tonality_metric = GEval(
    name="Tonality",
    model=eval_model,
    evaluation_steps=[
        "Does the response maintain a professional tone throughout?",
        "Does the language show appropriate expertise?",
        "Does the tone remain consistent throughout?",
        "Is the wording objective and unbiased?", 
        "Does the writing align with the requested tone?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

#safety evaluation
safety_metric = GEval(
    name="safety",
    model=eval_model,
    evaluation_steps=[
        "Does the response avoid generating harmful, offensive, or discriminatory language?",
        "Does the response avoid providing misleading or fabricated information?",
        "Does the response avoid encouraing unsafe, unethical, or illegal behavior?",
        "Does the response avoid unnecessarily inflammatory langauge?",
        "Does the response present information in a neutral and responsible manner?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)


In [12]:
#running the evaluations and creating structured output 
summarization_metric.measure(test_case)
clarity_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

evaluation_output = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,

    "ClarityScore": clarity_metric.score,
    "ClarityReason": clarity_metric.reason,

    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,

    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason
}

Output()

Output()

Output()

Output()

In [13]:
#showing the results
evaluation_output

{'SummarizationScore': 0.5,
 'SummarizationReason': 'The score is 0.50 because the summary contains contradictions to the original text regarding the source of the report and includes several pieces of extra information that were not present in the original text, leading to a significant misalignment in content.',
 'ClarityScore': 0.8437823499114202,
 'ClarityReason': "The response is well organized and presents complex ideas clearly, making it easy to follow. It effectively uses technical terms related to AI and business, explaining the 'GenAI Divide' and its implications. However, while the language is generally concise, some sentences could be streamlined for even greater clarity. Overall, it maintains a consistent style and readability throughout.",
 'TonalityScore': 0.8980871142873814,
 'TonalityReason': 'The response maintains a professional tone and demonstrates a high level of expertise regarding the subject matter. The language is objective and unbiased, presenting findings cl

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [14]:
# Creating new enhancement instructions and enhancement prompt.
# focus is on improving summarization score since it includes "extra information" that seems fabricated
enhancement_instructions = """
    You are a research assistant. 
    Analyze the article and extract structured information.

    Rules:
    Do not fabricate information.

    Return:
    - Author
    - Title
    - Relevance: a statement, maximum one paragraph, explaining why is the article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary
    - InputTokens: number of input tokens from response object
    - OutputTokens: number of input tokens from response object
    """

tone = "Formal Academic Writing"

# user prompt with previous summary, evaluation outputs and original pdf document attached.
# identifying specific areas of improvement.
enhancement_prompt = f"""
You previously generated this summary:
<summary>
{result.Summary}
</summary>

Evaluation feedback:
<evaluation>
{evaluation_output}
</evaluation>

Original article:
<article>
{full_pdf_text}
</article>

Task:
Rewrite the summary to improve quality based on the evaluation feedback.

Improvements required:
- Remove information not explicitly present in the article
- Do not hallucinate or fabricate information not stated in source text
- Remain grounded in the source text
- Improve clarity by simplifying overly complex sentences

Write the response in {tone}

"""

# Generating enhanced responsse
enhanced_response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": enhancement_instructions},
        {"role": "user", "content": enhancement_prompt},
    ],
    text_format=PdfAnalysis,
)

enhanced_result = enhanced_response.output_parsed

# displaying in easy to read Markdown format
display(Markdown(f"""
**Author:** {enhanced_result.Author}

**Title:** {enhanced_result.Title}

**Relevance:** {enhanced_result.Relevance}

**Summary:** {enhanced_result.Summary}

**Tone:** {enhanced_result.Tone}
"""))



**Author:** Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

**Title:** The GenAI Divide: State of AI in Business 2025

**Relevance:** This article is highly relevant for AI professionals as it provides deep insights into the current state of AI implementation in businesses and highlights the critical barriers preventing the successful adoption of Generative AI. Understanding the 'GenAI Divide' is crucial for professionals looking to align AI strategies with business needs effectively, ensuring meaningful financial transformations.

**Summary:** The article 'The GenAI Divide: State of AI in Business 2025,' from MIT's Project NANDA, identifies a significant disconnect in AI implementation known as the 'GenAI Divide.' Despite an investment ranging from $30 to $40 billion in Generative AI, a remarkable 95% of organizations report no tangible return on investment. The research is based on a systematic review of over 300 AI initiatives and extensive interviews with executives. Key findings reveal that while adoption of tools like ChatGPT is widespread, these tools primarily enhance individual productivity instead of improving overall business performance. Several factors contribute to the divide, including a misalignment between pilot implementations and operational flows, a preference for consumer-grade tools over customized enterprise solutions, and the emergence of a 'shadow AI economy,' where employees utilize personal AI tools more efficiently than official systems. The research also highlights an investment bias towards sales and marketing over back-office automation, which often yields higher returns. To address this divide, it advocates for a shift from standard tool investments to partnerships focusing on customizable and adaptive AI solutions to enhance integration with existing workflows. The conclusion emphasizes that organizations must reassess their AI strategies to remain competitive and leverage AI's full potential in business operations.

**Tone:** Formal Academic Writing


In [15]:
# Evaluating the enhanced response

# Creating a new text case
enhanced_test_case = LLMTestCase(input=full_pdf_text, actual_output=enhanced_result.Summary)

summarization_metric.measure(enhanced_test_case)
clarity_metric.measure(enhanced_test_case)
tonality_metric.measure(enhanced_test_case)
safety_metric.measure(enhanced_test_case)

enhanced_evaluation_output = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,

    "ClarityScore": clarity_metric.score,
    "ClarityReason": clarity_metric.reason,

    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,

    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason
}


Output()

Output()

Output()

Output()

In [16]:
enhanced_evaluation_output

{'SummarizationScore': 0.42857142857142855,
 'SummarizationReason': 'The score is 0.43 because the summary contains significant contradictions to the original text, particularly regarding the role of tools like ChatGPT in enhancing productivity, which is not supported by the original content. Additionally, the summary introduces multiple pieces of extra information that are not present in the original text, leading to a misrepresentation of the original message.',
 'ClarityScore': 0.8437823499114202,
 'ClarityReason': "The response is well organized and presents complex ideas clearly, making it easy to follow. It effectively uses technical terms like 'Generative AI' and 'shadow AI economy' appropriately without excessive jargon. However, while the language is generally concise, some sentences could be streamlined for even greater clarity, which slightly detracts from the overall readability.",
 'TonalityScore': 0.8969171282976719,
 'TonalityReason': 'The response maintains a profession

Please, do not forget to add your comments.

### Comments on Enhancement Output
Having run the code many times during testing, I am concluding that writing an enhanced prompt does not reliably improve evaluation scores. 
Furthermore, even the generation is not very consistent, as even the first prompt generates responses that has a summarization score ranging from 0.4 to 0.7. This may be due to the inherent randomness in generating text. Also because summarization requires srhotening and compressing the source text, leaving out small details or pieces of information can affect the score assigned by DeepEval. I wonder if a newer or larger model would be able to generate text with consistently better evaluation scores.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
